In [1]:
%cd /drive2/ryusejong/LFF
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "7"
import json 
import time 
import re
import random
import numpy as np 
from tqdm.auto import tqdm
from util.utils import set_seed, read_data, save_result, get_answer_from_text, chat_huggingface, contruct_conversation
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel
from gritlm import GritLM


seed = 42
set_seed(seed)

/drive2/ryusejong/LFF


/drive2/ryusejong/miniconda3/envs/llm1/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Sample Failure

In [2]:
path_input = "dataset/GSM8K_train_seed42_portion0.1.jsonl"
#path_result = "output/GSM8K_Llama-3-8B-Instruct_zeroshot_CoT_train_512.jsonl"
path_result = "output/GSM8K_Llama-3-8B-Instruct_zeroshot_CoT_train_512_seed42_portion0.1.jsonl"

data = read_data(path_input)
results = read_data(path_result)

sample_portion = 1.0  # Set the portion of the dataset 
sample_k = int(np.floor(sample_portion * len(results)))
sample_indices = np.sort(np.random.choice(np.arange(len(results)), size=sample_k, replace=False))

path_output = f"failure/GSM8K_Llama-3-8B-Instruct_zeroshot_CoT_train_512_seed42_portion0.1.jsonl"
print(f"{len(sample_indices)} samples")

FileNotFoundError: [Errno 2] No such file or directory: 'output/GSM8K_Llama-3-8B-Instruct_zeroshot_CoT_train_512_seed42_portion0.1.jsonl'

In [ ]:
i = 0
fail_list = []
for index in tqdm(sample_indices):
    answer = results[index].get("answer", "")
    pred_ans = results[index].get("pred_ans", "")
    
    if answer != pred_ans:
        fail_list.append({"index": int(index),
                          "question": data[results[index].get("index", 0)]["question"],
                          "answer": answer,
                          "reasoning": data[results[index].get("index", 0)]["reasoning"],
                          "fail_answer": pred_ans,
                          "fail_reasoning": results[index].get("A", "").get("content", "")
                        })
        i += 1

print(f"Total samples: {len(sample_indices)}, Failures: {len(fail_list)}, Percentage: {len(fail_list) / len(sample_indices) * 100:.2f}%")

for fail in fail_list:
    save_result(fail, path_output)

100%|██████████| 747/747 [00:00<00:00, 572159.44it/s]

Total samples: 747, Failures: 123, Percentage: 16.47%


## 2. Get Advice from GPT-o3

### Error Extraction Prompt

Name of failure file $\rightarrow$ "PROBLEMs.jsonl", then extract and analysis error from failure using following prompt.

__< Used Prompt ( + failure output .jsonl file) >__

![내 이미지](./img/error_extract_prompt.png "설명 텍스트")

PROBLEM STRUCTURE:
{
	"index": index of the each PROBLEM,
	"question": a mathematical problem that a model cannot solve and attempts to extract mathematical reasoning patterns,
	"answer": correct answer to the each question,
	"reasoning": a reasoning process that leads to the correct answer to the each question,
	"fail_answer": model's wrong answer of the each question,
	"fail_reasoning": the reasoning process where the model gives the wrong answer to the each question
}

ERROR STRUCTURE:
{
	"index": index of the each PROBLEM,
	"type": a type of error occurred in the fail_reasoning, the process of solving the question by referring to the correct reasoning of each PROBLEM,
	"logic": detailed case of logic where an error occured in fail_reasoning of each PROBLEM,
	"format": general format of above each logic where an error occurred
}

EXAMPLE:
{"index": 169, "type": "unit scaling error", "logic": "Converted consumption from 90 kg/6 months to 15 kg/month correctly, but then multiplied by 2 again for 'two six‑month periods' before annual scaling, double‑counting the factor and overstating the yearly total.", "format": "Rate per longer period → rate per shorter period → multiplied again by (longer/shorter) before final scaling, causing double‑counting and inflated result."}
{"index": 743, "type": "arithmetic multiplication error", "logic": "Computed 245 gallons/mile × 400 miles as 980 instead of 98,000, dropping two zeros and misplacing place value, leading to gross underestimation.", "format": "When multiplying multi‑digit numbers, zeros or place value are omitted, producing a result smaller by a power of 10."}

PROBLEMs whose components follow the PROBLEM STRUCTURE format is given as a .JSONL file. Perform the INSTRUCTION below.

INSTRUCTION:
For each PROBLEM component in PROBLEMs, analyze it, extract core logic and strategies, refer to EXAMPLE and create universal ERROR following the ERROR STRUCTURE format.
For each PROBLEM, please think deeply and ensure that your response is highly concise and structured, so that general ERROR can be extracted from the specific PROBLEM.
Crucially, make sure the LLM's output aligns with our ERROR STRUCTURE and return the all ERROR outputs in a ERRORs.JSONL file.
(Do not use U+202f.) 
(Think and reason for about 6 minutes.)

### Error Generalization Prompt

![내 이미지](./img/error_generalize_prompt.png "설명 텍스트")

ERROR STRUCTURE:
{
	"index": index of the each ERROR,
	"type": a type of error occurred in the logic,
	"logic": detailed case of logic where an error occured for each ERROR,
	"format": general format of above each logic where an error occurred
}

COMMONERROR STRUCTURE:
{
	"type": a generalized  representive type of ERRORs which are classified in this COMMONERROR cluster by referring to the type, logic, and format of each ERROR,
	"format": general format of generalized COMMONERROR
}

ERRORs whose components follow the ERROR STRUCTURE format is given as a .JSONL file. Perform the INSTRUCTION below.

INSTRUCTION:
For each ERROR component in ERRORs, analyze it, extract the core error and its reason, then cluster the ERRORs into a non-overlapping set of groups with similar error characteristics.
Instead of specifying a fixed number (e.g., 50 or 100), please cluster the ERRORs into the optimal number of groups based on their semantic and logical similarity.
For each group, create a COMMONERROR that follows the given COMMONERROR STRUCTURE.
Ensure the COMMONERRORs are highly concise and generalizable representations of their respective group.
Do not create unclassified error group and determine the type of group as much as possible.
Output the final COMMONERRORs in a JSONL file named COMMONERRORs.JSONL.
(Do not use U+202f.)
(Think and reason for about 6 minutes.)

### New Error Extract Prompt

############################################################
TASK SUMMARY
############################################################
Input file :  FAILUREs.jsonl        (one JSON object per line)
Output file:  ERRORs.jsonl          (one JSON object per line)

For each input object produce an ERROR object with:
  {"index": <int>, "description": <string>, "instruction": <string>}

Key order must be exactly: index, description, error_tag, instruction.

############################################################
INPUT FIELDS  (read only)
############################################################
index          : integer  – unique ID
question       : string   – problem text
answer         : string | number – correct final answer
reasoning      : string   – ideal solution
fail_answer    : string | number – wrong answer produced
fail_reasoning : string   – flawed reasoning

############################################################
REFLECTION RULE
############################################################
  • **Run an internal reflection loop exactly three times** before finalising description, error_tag, and instruction for a record.  
  • Each pass silently checks:  
    – Is error_tag the minimal neutral root-cause?  
    – Does description fully and correctly explain the divergence?  
    – Is instruction tag-specific, ONE sentence, generic, and non-duplicate?  
  • Do **not** output any reflection thoughts or intermediate drafts.

############################################################
OUTPUT FIELDS  (write)
############################################################
description, error_tag
  • Review and understand correct approach (from “reasoning” + “answer”).  
  • Compare fail_reasoning to correct reasoning.
  • Manually analyze and explain in detail why “fail_reasoning” happens instead of correct approach
       e.g. lack of understanding of question situation, incorrect calculation, unintended addional/missing reasoning steps, missing some conditions. etc.  
  • Choose root cause of "fail_reasoning" and summarize in 1-3 neutral words → error_tag
         Free to create new proper tag, choose from the examples below, or choose from seen_tags.
       e.g. **[multiple/missing-application]*, **[lack-of-understanding]**, **[missing-condition]**, **[unit-error]**, **[arithmetic-operation-error]**, **[sign-error]**, **[inverse-ratio]**, **[truncation]**, **[double-counting]**, **[counting-error]**, **[comparison-misread]**, **[double-subtraction]**, **[substitution-error]**, **[condition-misinterpret]** etc.  
       Tag = 1-3 neutral words, no units or concrete nouns.

  • If fail_answer == 10086100100.0
      – fail_reasoning cuts off AND has no `## number ##` → error_tag = **Truncation**.   
      – a `## number ##` appears but its value is unusable → error_tag = **Invalid-numeric**.

instruction   (single, fully generic sentence)
  • Give generic advice about how to fix the detected error pattern.  
  • **Test the drafted advice against the reasoning steps; if the advice would not actually prevent the same error, regenerate a new one and re-test until it passes.**
  • If truncation → remind the model to keep reasoning concise so the final value appears in ## number ## format before the token limit.  
  • If invalid-numeric → remind the model to output the final value alone using a correct ## number ## pattern.
  • Duplicate wording is allowed **only when the leading [tag] matches a previous line**; otherwise the sentence must differ.  
  • No concrete units or context words (hours, kg, dollars, trains).  
  • Allowed neutral terms: value, variable, rate, interval, subtotal, result, equation, verify, etc.

############################################################
SELF-CHECK (run before writing each line)
############################################################
☐ instruction is ONE sentence  
☐ if the tag is **new**, instruction is not identical to any earlier instruction  
☐ instruction contains no forbidden unit/context words  
☐ keys in order: index, description, error_tag, instruction  
☐ output is valid JSON on one line (no trailing comma, no blank lines)

############################################################
ALGORITHM
############################################################
seen_tags       = set()      # which tags have appeared so far
seen_instructions = set()        # cache of previous line’s instruction

for each line in FAILUREs.jsonl:
    parse → data
    
    # ── build description ───────────────────────────────
    description, error_tag = (follow OUTPUT rules)

    # ── build instruction ──────────────────────────
    instruction = (follow OUTPUT rules)

    # ── duplication guard ───────────────────────────────
    if error_tag not in seen_tags:
        while instruction in seen_instructions:
            instruction = (follow OUTPUT rules)

    # ── self-check bullets ─────────────────────────────
    run SELF-CHECK; revise until all pass

    # ── reflection ─────────────────────────────
    run REFLECTION

    # ── emit ────────────────────────────────────────────
    write {"index": data.index,
           "description": description,
           "error_tag": error_tag,
           "instruction": instruction}  # one-line JSON

    seen_tags.add(error_tag)
    seen_instructions.add(instruction)


## 3. Extract Fail Question Latent Vector


Using Llama-3-8B-Instruct model

In [2]:
HUGGINGFACE_TOKEN = "xxx"

#model_path = "jinaai/jina-embeddings-v3"
#model_path = "GritLM/GritLM-7B"
model_path = "reasonir/ReasonIR-8B"
device = "cuda:0" if torch.cuda.is_available() else "cpu"

model = AutoModel.from_pretrained(
    model_path,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16
)
"""
model = GritLM(
    model_path,
    torch_dtype=torch.bfloat16
)
"""
model.eval()
model = model.to(device)

Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00, 80.71it/s]


In [3]:
def gritlm_instruction(instruction):
    return "<|user|>\n" + instruction + "\n<|embed|>\n" if instruction else "<|embed|>\n"

In [3]:
#path_tensor = f"memory/GSM8K_Llama-3-8B-Instruct_zeroshot_CoT_train_seed{seed}_portion{sample_portion}.pt"
#path_revision = f"memory/GSM8K_Llama-3-8B-Instruct_zeroshot_CoT_train_seed{seed}_portion{sample_portion}.jsonl"
#path_tensor = f"memory/FAILUREs_reasonir.pt"
#path_tensor = f"memory/FAILURE_ERRORs_reasonir.pt"
path_tensor = f"memory/COMMONERRORs_reasonir.pt"
path_failure = f"failure/FAILUREs.jsonl"
#path_error = f"memory/ERRORs.jsonl"
path_error = f"memory/COMMONERRORs.jsonl"

failure_list = read_data(path_failure)
error_list = read_data(path_error)

# extract the last latent representation(vector) from the question
failure_embeddings = []

for failure, error in tqdm(zip(failure_list, error_list)):
    #failure_text = failure["question"] + "\n" + failure["fail_reasoning"]
    #failure_text = failure["question"] + "\n" + failure["fail_reasoning"] + "\n" + error["description"]
    failure_text = "[" + error["error_tag"] + "]: " + error["description"]
    print(failure_text)
    
    with torch.no_grad():
        # jinaai
        #error_embedding = model.encode(failure_text, task="text-matching")
        # GritLM
        #error_embedding = model.encode(failure_text, instruction=gritlm_instruction(""), return_tensors="pt")
        # ReasonIR
        error_embedding = model.encode(failure_text, instruction="")
        
    failure_embeddings.append(error_embedding)
    
# Save the question vectors to a tensor file
torch.save(torch.tensor(failure_embeddings), path_tensor)

0it [00:00, ?it/s]

[No error]: No error detected in the reasoning.


5it [00:01,  4.54it/s]

[Invalid-input]: Input does not conform to expected format or type, such as a string where a number is required.
[Division-by-zero]: Attempts to divide by zero, which is undefined.
[Out-of-bounds]: Accesses an index or value outside the defined range of a list or array.
[Type-mismatch]: Combines incompatible types, such as adding a string to a number.
[Syntax-error]: Contains invalid syntax that prevents execution, such as missing parentheses or incorrect operators.
[Logic-error]: The logic of the solution is flawed, leading to incorrect results without syntax errors.
[Invalid-numeric]: Generates a token that is not a valid numeral, forcing a fallback or sentinel value.


13it [00:01, 12.89it/s]

...

21it [00:01, 20.79it/s]

[double-counting]: Adds the same subtotal more than once, inflating the final result.
[double-scaling]: Applies the same conversion or scaling factor twice.
[double-subtraction]: Removes the same portion twice, producing an unduly small value.
[equation-setup-error]: Builds equations that do not reflect the stated relationships.
[inverse-ratio]: Uses the reciprocal of a required ratio.
[misordered-calculation]: Executes steps in the wrong sequence, so later operations act on an incorrect intermediate value.
[missing-condition]: Omits a stated requirement while solving.
[missing-division]: Fails to divide by a necessary count or rate when computing an average or per-unit value.


30it [00:02, 13.92it/s]

[missing-update]: Does not adjust an intermediate subtotal after a stated change, so an outdated value persists.
[overlap-error]: Ignores intersection between groups, leading to over- or under-count.
[rounding-error]: Rounds too early or incorrectly, shifting the intermediate value enough to change the final result.
[substitution-error]: Inserts the wrong variable or value into an equation.
[subtraction-error]: Swaps minuend and subtrahend or miscomputes the difference.
[unit-error]: Uses inconsistent units or forgets to convert before combining values.



/tmp/ipykernel_36306/891056112.py:33: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  torch.save(torch.tensor(failure_embeddings), path_tensor)


In [4]:
failure_embeddings = torch.load(path_tensor)
print(type(failure_embeddings), failure_embeddings.shape, failure_embeddings.dtype)

<class 'torch.Tensor'> torch.Size([30, 4096]) torch.float32
